# Extract PDF text, then copy originals

1. Reads every `.pdf` file in a source folder.
2. Extracts its text (handling two-column layouts) and saves a matching `.txt` file into a destination folder.
3. Copies **every file** from the source folder (not just PDFs) into the destination folder. Originals stay in the source folder.

Runs this same two-step process for **multiple folder pairs** — currently the
Species RAG corpus and the Regional status tool corpus — so both get
extracted and mirrored in one run.

Uses `pdfplumber` for extraction. Multi-column pages are detected automatically (by finding a vertical gap with no text near the middle of the page) and each column is extracted separately, top-to-bottom, left column then right column, so text no longer gets scrambled across columns. Falls back to `pypdf` if a page fails to parse entirely. Run the install cell once if `pdfplumber` isn't already available.

In [1]:
from pathlib import Path
import json
import shutil

import pdfplumber
from pypdf import PdfReader

# Minimum width of the empty vertical gap that separates two columns, as a fraction
# of the page width. Lower this if a two-column layout isn't being detected;
# raise it if a single-column page is incorrectly being split in two.
MIN_COLUMN_GAP_FRACTION = 0.03
MIN_COLUMN_GAP_POINTS = 8

def find_column_split(page) -> float | None:
    """Return the x-coordinate of the gap between two text columns, or None for single-column pages.

    Looks for the widest horizontal gap between consecutive words (by x-center) that falls
    roughly in the middle of the page. If that gap is wide enough, it's treated as the
    gutter between two columns.

    Coordinates are computed relative to the page's own bounding box (page.bbox), not
    assumed to start at x=0. Some PDFs (e.g. pages exported from a two-page spread) have
    a mediabox that doesn't start at the origin, and even/odd pages can be offset
    differently -- using page.width alone would silently break half the pages.
    """
    words = page.extract_words()
    if not words:
        return None

    page_x0, _, page_x1, _ = page.bbox
    page_width = page_x1 - page_x0
    centers = sorted((word['x0'] + word['x1']) / 2 for word in words)
    best_gap = 0.0
    split_x = None

    for previous_center, current_center in zip(centers, centers[1:]):
        gap = current_center - previous_center
        midpoint = (previous_center + current_center) / 2
        # Only consider gaps roughly in the middle of the page (not margins).
        if page_x0 + 0.2 * page_width < midpoint < page_x0 + 0.8 * page_width and gap > best_gap:
            best_gap = gap
            split_x = midpoint

    min_gap = max(MIN_COLUMN_GAP_FRACTION * page_width, MIN_COLUMN_GAP_POINTS)
    return split_x if best_gap > min_gap else None

def extract_page_text(page) -> str:
    """Extract a page's text, splitting into left/right columns first if a column gap is found."""
    split_x = find_column_split(page)
    if split_x is None:
        return (page.extract_text() or '').strip()

    page_x0, page_top, page_x1, page_bottom = page.bbox
    left_column = page.crop((page_x0, page_top, split_x, page_bottom))
    right_column = page.crop((split_x, page_top, page_x1, page_bottom))
    left_text = (left_column.extract_text() or '').strip()
    right_text = (right_column.extract_text() or '').strip()
    return '\n\n'.join(text for text in (left_text, right_text) if text)

def extract_pdf_text(pdf_path: Path) -> str:
    """Extract text from a PDF, page by page, handling two-column layouts.

    Falls back to pypdf (single-column reading order) if pdfplumber can't get any text at all.
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            try:
                text = extract_page_text(page)
            except Exception as error:
                print(f'  Warning: column-aware extraction failed on page {page_number} ({error}); using plain extraction.')
                try:
                    text = (page.extract_text() or '').strip()
                except Exception:
                    text = ''
            pages_text.append((page_number, text))

    # Fallback for PDFs pdfplumber could not read at all.
    if not any(text for _, text in pages_text):
        reader = PdfReader(pdf_path)
        pages_text = []
        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = (page.extract_text() or '').strip()
            except Exception:
                text = ''
            pages_text.append((page_number, text))

    return '\n\n'.join(f'--- Page {number} ---\n{text}' for number, text in pages_text if text)

def carry_forward_source_metadata(pdf_path: Path, text_file: Path) -> None:
    """Preserve citation metadata across the PDF -> txt step.

    Two cases:
    1. An upstream '<pdf_path>.meta.json' sidecar already exists (written by
       02_BooksArticles-Extractor / 02_Libgen-Extractor with real resolved
       title/author/url/etc.) -- copy it forward under the new '.txt' name
       so 04_ChunkEmbedChromaRetrieve can still find it.
    2. No sidecar exists (e.g. a PDF you dropped in by hand) -- fall back to
       whatever the PDF's own embedded document-info dictionary provides
       (Title/Author/CreationDate), which is genuine data read from the file
       itself, not a guess. Always records the original PDF filename too,
       so there's at least a trail back to the source file.
    """
    meta_out_path = text_file.with_suffix(text_file.suffix + '.meta.json')

    upstream_meta_path = pdf_path.with_suffix(pdf_path.suffix + '.meta.json')
    if upstream_meta_path.exists():
        shutil.copy2(str(upstream_meta_path), str(meta_out_path))
        return

    doc_info = {}
    try:
        reader = PdfReader(pdf_path)
        info = reader.metadata or {}
        doc_info = {
            'title': info.get('/Title'),
            'author': info.get('/Author'),
            'creation_date': info.get('/CreationDate'),
            'producer': info.get('/Producer'),
        }
    except Exception:
        pass

    payload = {k: v for k, v in doc_info.items() if v}
    payload['source_type'] = payload.get('source_type', 'pdf_upload')
    payload['original_filename'] = pdf_path.name
    meta_out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')


In [2]:
# Each entry: (label, SOURCE_FOLDER, DEST_FOLDER). Files in SOURCE_FOLDER are
# never deleted or moved; DEST_FOLDER receives extracted .txt files plus a
# copy of every original file.

FOLDER_PAIRS = [
    (
        "Species RAG",
        Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' / 'a.Raw Extraction Documents - Species RAG',
        Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' / 'b.Text from Extraction Documents - Species RAG',
    ),
    (
        "Regional status tool",
        Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' / 'a.Raw Extraction Documents - Regional status tool',
        Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' / 'b.Text from Extraction Documents - Regional status tool',
    ),
]


In [3]:
def process_folder_pair(label: str, source_folder: Path, dest_folder: Path) -> None:
    """Run both steps (extract text, then copy every file) for one source/dest folder pair."""
    print(f'=== {label} ===')
    dest_folder.mkdir(parents=True, exist_ok=True)

    # Step 1: extract text from every PDF in source_folder and save it as .txt in dest_folder.
    pdf_files = sorted(source_folder.glob('*.pdf'))
    if not pdf_files:
        print(f'No PDF files found in: {source_folder}')

    for pdf_path in pdf_files:
        try:
            text = extract_pdf_text(pdf_path)
        except Exception as error:
            print(f'Failed to read {pdf_path.name}: {error}')
            continue

        if not text:
            print(f'No extractable text (likely scanned/image-only): {pdf_path.name}')
            continue

        text_file = dest_folder / f'{pdf_path.stem}.txt'
        text_file.write_text(text, encoding='utf-8')
        print(f'Saved: {text_file}')
        carry_forward_source_metadata(pdf_path, text_file)

    # Step 2: copy every file from source_folder into dest_folder (PDFs and anything else).
    # Originals remain in source_folder untouched.
    for source_path in sorted(source_folder.iterdir()):
        if not source_path.is_file():
            continue

        destination_path = dest_folder / source_path.name
        if destination_path.exists():
            destination_path = dest_folder / f'{source_path.stem}_copy{source_path.suffix}'

        shutil.copy2(str(source_path), str(destination_path))
        print(f'Copied to: {destination_path}')

    print()


for label, source_folder, dest_folder in FOLDER_PAIRS:
    process_folder_pair(label, source_folder, dest_folder)


=== Species RAG ===
No PDF files found in: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/a.Raw Extraction Documents - Species RAG
Copied to: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text from Extraction Documents - Species RAG/open_library_A_Handbook_of_the_British_Flora_copy.txt
Copied to: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text from Extraction Documents - Species RAG/open_library_A_Handbook_of_the_British_Flora.txt.meta_copy.json
Copied to: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text from Extraction Documents - Species RAG/open_library_A_History_of_British_Birds_William_Yarrell_copy.txt
Copied to: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text from Extraction Documents - Species RAG/open_library_A_History_of_British_Birds_William_Yarrell.txt.meta_copy.json
Copied to: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text 